# 03 - Data Preprocessing

## Objective

The objective of this notebook is to clean and preprocess the telecom customer dataset before machine learning model development. This includes removing unnecessary features, identifying data leakage, handling missing values, preparing categorical and numerical features, and producing a model-ready dataset.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [3]:
df = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")

print("Dataset loaded successfully!")
df.head()

Dataset loaded successfully!


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
df_original = df.copy()

## Data Type Correction

The **Total Charges** feature should be numerical. It is converted from object to numeric format before model development.

In [5]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")

In [6]:
missing_total_charges = df[df["Total Charges"].isna()]

print("Missing Total Charges rows:", missing_total_charges.shape[0])

missing_total_charges[
    ["Tenure Months", "Monthly Charges", "Total Charges"]
]

Missing Total Charges rows: 11


,Tenure Months,Monthly Charges,Total Charges
2234,0,52.55,NaN
2438,0,20.25,NaN
2568,0,80.85,NaN
2667,0,25.75,NaN
2856,0,56.05,NaN
4331,0,19.85,NaN
4687,0,25.35,NaN
5104,0,20.00,NaN
5719,0,19.70,NaN
6772,0,73.35,NaN


In [7]:
print("Original dataset has been backed up successfully.")

Original dataset has been backed up successfully.


In [8]:
print(f"Dataset Shape: {df.shape}")

Dataset Shape: (7043, 33)


## Feature Review and Data Leakage Analysis

Before preprocessing, each feature will be evaluated based on its predictive value, availability at prediction time, and potential risk of data leakage.

In [9]:
df.columns.tolist()

['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

In [10]:
for column in df.columns:
    print(f"{column}: {df[column].nunique()} unique values")

CustomerID: 7043 unique values
Count: 1 unique values
Country: 1 unique values
State: 1 unique values
City: 1129 unique values
Zip Code: 1652 unique values
Lat Long: 1652 unique values
Latitude: 1652 unique values
Longitude: 1651 unique values
Gender: 2 unique values
Senior Citizen: 2 unique values
Partner: 2 unique values
Dependents: 2 unique values
Tenure Months: 73 unique values
Phone Service: 2 unique values
Multiple Lines: 3 unique values
Internet Service: 3 unique values
Online Security: 3 unique values
Online Backup: 3 unique values
Device Protection: 3 unique values
Tech Support: 3 unique values
Streaming TV: 3 unique values
Streaming Movies: 3 unique values
Contract: 3 unique values
Paperless Billing: 2 unique values
Payment Method: 4 unique values
Monthly Charges: 1585 unique values
Total Charges: 6530 unique values
Churn Label: 2 unique values
Churn Value: 2 unique values
Churn Score: 85 unique values
CLTV: 3438 unique values
Churn Reason: 20 unique values


## Feature Selection Strategy

Each feature is evaluated before model development.

Features are classified into three categories:

- Features to remove
- Features to keep
- Features requiring further evaluation

The objective is to eliminate unnecessary variables and prevent data leakage while preserving valuable predictive information.

## Removing Unnecessary Features

The following features are removed because they are identifiers, constant values, duplicated information, or introduce potential data leakage into the machine learning model.

In [11]:
columns_to_drop = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude",
    "Churn Label",
    "Churn Score",
    "Churn Reason"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

print(f"Remaining features: {df.shape[1]}")
df.head()

Remaining features: 21


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,5372
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,5003
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,5340


## Identifying Numerical and Categorical Features

Before applying preprocessing techniques, the remaining features are separated into categorical and numerical variables. This separation is required because categorical features will be encoded using One-Hot Encoding, while numerical features will remain unchanged during preprocessing.

In [12]:
categorical_features = df.select_dtypes(include="object").columns.tolist()
numerical_features = df.select_dtypes(exclude="object").columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

Numerical Features:
['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Value', 'CLTV']


## Defining Features and Target Variable

The target variable (`Churn Value`) is separated from the input features. The remaining variables constitute the feature matrix that will be used for model training and evaluation.

In [13]:
X = df.drop("Churn Value", axis=1)
y = df["Churn Value"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (7043, 20)
Target Shape: (7043,)


In [14]:
df["Total Charges"] = df["Total Charges"].fillna(0)

print(
    "Remaining missing Total Charges values:",
    df["Total Charges"].isna().sum()
)

Remaining missing Total Charges values: 0


## Handling Missing Total Charges Values

After converting `Total Charges` to a numerical data type, 11 missing values were identified. These records belong to customers with zero tenure, indicating that no cumulative charge has yet been generated. Therefore, the missing values are replaced with **0**, which is consistent with the business meaning of the data.

In [15]:
print("Total Charges missing values:", df["Total Charges"].isna().sum())

Total Charges missing values: 0


In [16]:
df["Total Charges"] = df["Total Charges"].fillna(0)

print(
    "Remaining missing Total Charges values:",
    df["Total Charges"].isna().sum()
)

Remaining missing Total Charges values: 0


In [17]:
categorical_features = df.select_dtypes(include="object").columns.tolist()
numerical_features = df.select_dtypes(exclude="object").columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

Numerical Features:
['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Value', 'CLTV']


## Preprocessing Progress Summary

### Completed Tasks

- Removed identifier and constant features.
- Removed features that may introduce data leakage.
- Corrected data types.
- Verified that the dataset contains no missing values.
- Identified categorical and numerical features.
- Evaluated geographic variables and removed high-cardinality location features.



In [18]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (7043, 21)

Columns:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value', 'CLTV']


## Defining Features and Target

The target variable is separated from the input features before splitting the dataset into training and testing sets.

In [19]:
X = df.drop("Churn Value", axis=1)
y = df["Churn Value"]

print("Feature Matrix:", X.shape)
print("Target Vector:", y.shape)

Feature Matrix: (7043, 20)
Target Vector: (7043,)


## Train-Test Split

The cleaned dataset is divided into training and testing subsets. The training set will be used to fit preprocessing steps and machine learning models, while the testing set will be reserved for final evaluation.

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [21]:
print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)

print("Training Target   :", y_train.shape)
print("Testing Target    :", y_test.shape)

Training Features : (5634, 20)
Testing Features  : (1409, 20)
Training Target   : (5634,)
Testing Target    : (1409,)


## Encoding Strategy

Machine learning algorithms require numerical input. Therefore, categorical features are transformed into numerical representations using **One-Hot Encoding**. Numerical features remain unchanged and are passed directly to the model through the preprocessing pipeline.

In [22]:
categorical_features = X_train.select_dtypes(include="object").columns.tolist()
numerical_features = X_train.select_dtypes(exclude="object").columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

Numerical Features:
['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']


## Preprocessing Pipeline

A preprocessing pipeline is created using `ColumnTransformer`. Categorical features are transformed using **One-Hot Encoding**, while numerical features are passed through without modification. The preprocessing pipeline is fitted only on the training data to prevent data leakage.

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = X_train.select_dtypes(include="object").columns.tolist()
numerical_features = X_train.select_dtypes(exclude="object").columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

Numerical Features:
['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']


In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

In [25]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape :", X_test_processed.shape)

Processed Training Shape: (5634, 47)
Processed Testing Shape : (1409, 47)


## Creating Processed DataFrames

The transformed NumPy arrays are converted into pandas DataFrames using the feature names generated by the preprocessing pipeline.

In [26]:
feature_names = preprocessor.get_feature_names_out()

print("Number of Processed Features:", len(feature_names))
print(feature_names)

Number of Processed Features: 47
['categorical__Gender_Female' 'categorical__Gender_Male'
 'categorical__Senior Citizen_No' 'categorical__Senior Citizen_Yes'
 'categorical__Partner_No' 'categorical__Partner_Yes'
 'categorical__Dependents_No' 'categorical__Dependents_Yes'
 'categorical__Phone Service_No' 'categorical__Phone Service_Yes'
 'categorical__Multiple Lines_No'
 'categorical__Multiple Lines_No phone service'
 'categorical__Multiple Lines_Yes' 'categorical__Internet Service_DSL'
 'categorical__Internet Service_Fiber optic'
 'categorical__Internet Service_No' 'categorical__Online Security_No'
 'categorical__Online Security_No internet service'
 'categorical__Online Security_Yes' 'categorical__Online Backup_No'
 'categorical__Online Backup_No internet service'
 'categorical__Online Backup_Yes' 'categorical__Device Protection_No'
 'categorical__Device Protection_No internet service'
 'categorical__Device Protection_Yes' 'categorical__Tech Support_No'
 'categorical__Tech Support_No 

In [27]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Processed Training DataFrame:", X_train_processed_df.shape)
print("Processed Testing DataFrame :", X_test_processed_df.shape)

X_train_processed_df.head()

Processed Training DataFrame: (5634, 47)
Processed Testing DataFrame : (1409, 47)


,categorical__Gender_Female,categorical__Gender_Male,categorical__Senior Citizen_No,categorical__Senior Citizen_Yes,categorical__Partner_No,categorical__Partner_Yes,categorical__Dependents_No,categorical__Dependents_Yes,categorical__Phone Service_No,categorical__Phone Service_Yes,categorical__Multiple Lines_No,categorical__Multiple Lines_No phone service,categorical__Multiple Lines_Yes,categorical__Internet Service_DSL,categorical__Internet Service_Fiber optic,categorical__Internet Service_No,categorical__Online Security_No,categorical__Online Security_No internet service,categorical__Online Security_Yes,categorical__Online Backup_No,categorical__Online Backup_No internet service,categorical__Online Backup_Yes,categorical__Device Protection_No,categorical__Device Protection_No internet service,categorical__Device Protection_Yes,categorical__Tech Support_No,categorical__Tech Support_No internet service,categorical__Tech Support_Yes,categorical__Streaming TV_No,categorical__Streaming TV_No internet service,categorical__Streaming TV_Yes,categorical__Streaming Movies_No,categorical__Streaming Movies_No internet service,categorical__Streaming Movies_Yes,categorical__Contract_Month-to-month,categorical__Contract_One year,categorical__Contract_Two year,categorical__Paperless Billing_No,categorical__Paperless Billing_Yes,categorical__Payment Method_Bank transfer (automatic),categorical__Payment Method_Credit card (automatic),categorical__Payment Method_Electronic check,categorical__Payment Method_Mailed check,numerical__Tenure Months,numerical__Monthly Charges,numerical__Total Charges,numerical__CLTV
4626,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,35.0,49.20,1701.65,2782.0
4192,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,15.0,75.10,1151.55,4634.0
5457,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,13.0,40.55,590.35,2898.0
4717,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,26.0,73.50,1905.70,3596.0
4673,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,44.55,44.55,3408.0


## Combining Features with Target Variable

The processed feature matrices are combined with their corresponding target labels to create final machine learning datasets.

In [28]:
train_processed = X_train_processed_df.copy()
train_processed["Churn Value"] = y_train.values

test_processed = X_test_processed_df.copy()
test_processed["Churn Value"] = y_test.values

print(train_processed.shape)
print(test_processed.shape)

(5634, 48)
(1409, 48)


## Saving Processed Datasets

The processed training and testing datasets are saved for future machine learning experiments.

In [29]:
import os

os.makedirs("../data/processed", exist_ok=True)

train_processed.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

test_processed.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [30]:
print(train_processed.info())

<class 'pandas.core.frame.DataFrame'>
Index: 5634 entries, 4626 to 6017
Data columns (total 48 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   categorical__Gender_Female                             5634 non-null   float64
 1   categorical__Gender_Male                               5634 non-null   float64
 2   categorical__Senior Citizen_No                         5634 non-null   float64
 3   categorical__Senior Citizen_Yes                        5634 non-null   float64
 4   categorical__Partner_No                                5634 non-null   float64
 5   categorical__Partner_Yes                               5634 non-null   float64
 6   categorical__Dependents_No                             5634 non-null   float64
 7   categorical__Dependents_Yes                            5634 non-null   float64
 8   categorical__Phone Service_No                     

In [31]:
train_processed.head()

,categorical__Gender_Female,categorical__Gender_Male,categorical__Senior Citizen_No,categorical__Senior Citizen_Yes,categorical__Partner_No,categorical__Partner_Yes,categorical__Dependents_No,categorical__Dependents_Yes,categorical__Phone Service_No,categorical__Phone Service_Yes,categorical__Multiple Lines_No,categorical__Multiple Lines_No phone service,categorical__Multiple Lines_Yes,categorical__Internet Service_DSL,categorical__Internet Service_Fiber optic,categorical__Internet Service_No,categorical__Online Security_No,categorical__Online Security_No internet service,categorical__Online Security_Yes,categorical__Online Backup_No,categorical__Online Backup_No internet service,categorical__Online Backup_Yes,categorical__Device Protection_No,categorical__Device Protection_No internet service,categorical__Device Protection_Yes,categorical__Tech Support_No,categorical__Tech Support_No internet service,categorical__Tech Support_Yes,categorical__Streaming TV_No,categorical__Streaming TV_No internet service,categorical__Streaming TV_Yes,categorical__Streaming Movies_No,categorical__Streaming Movies_No internet service,categorical__Streaming Movies_Yes,categorical__Contract_Month-to-month,categorical__Contract_One year,categorical__Contract_Two year,categorical__Paperless Billing_No,categorical__Paperless Billing_Yes,categorical__Payment Method_Bank transfer (automatic),categorical__Payment Method_Credit card (automatic),categorical__Payment Method_Electronic check,categorical__Payment Method_Mailed check,numerical__Tenure Months,numerical__Monthly Charges,numerical__Total Charges,numerical__CLTV,Churn Value
4626,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,35.0,49.20,1701.65,2782.0,0
4192,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,15.0,75.10,1151.55,4634.0,0
5457,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,13.0,40.55,590.35,2898.0,0
4717,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,26.0,73.50,1905.70,3596.0,0
4673,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,44.55,44.55,3408.0,0


In [32]:
train_processed = X_train_processed_df.copy()
train_processed["Churn Value"] = y_train

test_processed = X_test_processed_df.copy()
test_processed["Churn Value"] = y_test

print("Train dataset shape:", train_processed.shape)
print("Test dataset shape :", test_processed.shape)

Train dataset shape: (5634, 48)
Test dataset shape : (1409, 48)


In [33]:
print(
    "Training missing values:",
    train_processed.isna().sum().sum()
)

print(
    "Testing missing values:",
    test_processed.isna().sum().sum()
)

Training missing values: 0
Testing missing values: 0


In [34]:
import os

os.makedirs("../data/processed", exist_ok=True)

train_processed.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

test_processed.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


## Saving Processed Datasets

The processed training and testing datasets are saved for future machine learning experiments.

In [35]:
import os

os.makedirs("../data/processed", exist_ok=True)

train_processed.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

test_processed.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [36]:
import pandas as pd

train_check = pd.read_csv("../data/processed/train_processed.csv")
test_check = pd.read_csv("../data/processed/test_processed.csv")

print(train_check.shape)
print(test_check.shape)

(5634, 48)
(1409, 48)
